# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 06 P7 site-shift analysis

**Permitted partitions: TRAIN + VALIDATION for fitting/selection; FINAL TEST — READ ONLY after all 12 fold selections and development replays pass.**

Choose a CPU runtime and **Runtime → Run all**. No SHA, stage or run-name entry is needed. Colab still requires Drive and `AIRSENSE_GITHUB_TOKEN` authorization. This repeats the frozen tuning plan for all 12 held-out stations: 756 learned-model fits. Expect substantially more computation than P6, potentially many sessions. Do not reduce the settings to speed up a final run.

Completed training units and predictions are saved to Drive, with compact evidence and checksums pushed to the single `results/p7-sites-01` branch. An interrupted in-flight unit can be repeated. After a browser disconnect, rerun cell 3 to attach to the existing worker. If Colab replaced the runtime, run all cells to restore checkpoints. Use only one runtime and never delete lock files. P5/P6 models are preserved. P7 completion still requires independent scientific review.

### Colab environment recovery (20 October 2026)
Use **Run all** in a CPU runtime with your original Drive account. The original run ID, scientific code, models, splits and budgets are retained. Setup restores required scientific dependency versions only. The worker records the actual environment and replays completed folds' development predictions before resuming unfinished training. Replay may take time; it does not retrain completed models. Do not run the original notebook simultaneously. Old worker logs are preserved. This recovery adapter has separate pinned code and a verified download checksum.

## 1. Automatic setup
The matching code revision is embedded in this notebook.

In [1]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = "7278ba95c88ea28e8861f59d0a6f8e097087c9d7"
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r-p7-7278ba95c88e"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-p5.txt"), "-e", str(ROOT)], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)

from airsense_r.artifacts import source_hash
if source_hash(ROOT) != "9c1237fbbad6e1ca6abe065bbbc9c7f91baf8e2381ed6c86ae50a8df7f6519d3":
    raise RuntimeError("Notebook/source mismatch; reopen the published P7 notebook")
print("P7 setup ready. Code revision and run settings are already configured.")

# Recovery adapter is separate from the immutable scientific source checkout.
RECOVERY_REVISION = "5b3cf77f52253f110059f0b00373bc794ddc2d86"
RECOVERY_SHA256 = "ae88dc6ab95d04c2a7708c08b16888cdc0338f8c90da7fbf2edc68f27857f877"
RECOVERY_SCRIPT = ROOT / ".tmp" / "p7-recovery" / RECOVERY_REVISION / "resume_p7_colab.py"
if not TEST_MODE:
    import hashlib
    import urllib.request
    url = (
        "https://raw.githubusercontent.com/Engr-Daniel/airsense-r/"
        + RECOVERY_REVISION + "/scripts/recovery/resume_p7_colab.py"
    )
    payload = urllib.request.urlopen(url, timeout=60).read()
    if hashlib.sha256(payload).hexdigest() != RECOVERY_SHA256:
        raise RuntimeError("Recovery download checksum mismatch")
    RECOVERY_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    RECOVERY_SCRIPT.write_bytes(payload)


Code revision: 7278ba95c88ea28e8861f59d0a6f8e097087c9d7
Workspace: /content/airsense-r-p7-7278ba95c88e
Verified package: /content/airsense-r-p7-7278ba95c88e/src/airsense_r/__init__.py
P7 setup ready. Code revision and run settings are already configured.


## 2. Connect persistent storage
Authorize your existing Drive and GitHub secret when Colab asks.

In [3]:
from airsense_r.colab import configure_github_secret
RUN_ID = "p7-sites-01"
if TEST_MODE:
    MODEL_DIRECTORY = Path(os.environ["AIRSENSE_P7_TEST_MODELS"])
    RUN_ID = os.environ["AIRSENSE_P7_TEST_RUN"]
    REMOTE = os.environ.get("AIRSENSE_P7_TEST_REMOTE", "unused-fixture")
else:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    MODEL_DIRECTORY = Path("/content/drive/MyDrive/AirSense-R/model-checkpoints")
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    REMOTE = REPOSITORY
if not (MODEL_DIRECTORY / "storage.json").is_file() and not TEST_MODE:
    raise FileNotFoundError("Existing AirSense-R Drive storage is unavailable; check the mounted account")
JOB_DIRECTORY = ROOT / ".tmp/notebook-jobs" / RUN_ID
print("Ready: automatic P7 run/resume. Use one Colab runtime at a time.")

# Preserve the failed worker's log; the same experiment writer lock is retained.
JOB_DIRECTORY = ROOT / ".tmp/notebook-jobs" / (RUN_ID + "-colab-recovery")
if not TEST_MODE:
    from airsense_r.notebook_job import status
    if status(JOB_DIRECTORY)["state"] != "running":
        print("Checking saved checkpoints and restoring required dependency versions. Please wait.", flush=True)
        preparation = subprocess.run(
            [sys.executable, str(RECOVERY_SCRIPT), "--prepare",
             "--root", str(ROOT), "--model-directory", str(MODEL_DIRECTORY),
             "--remote", REMOTE],
            cwd=ROOT, capture_output=True, text=True,
        )
        print(preparation.stdout, flush=True)
        if preparation.returncode:
            print(preparation.stderr, flush=True)
            raise RuntimeError("Recovery setup stopped; see the error above. No training started.")
print("Ready. Cell 3 will verify saved development predictions before resuming.")


Mounted at /content/drive
Ready: automatic P7 run/resume. Use one Colab runtime at a time.
Checking saved checkpoints and restoring required dependency versions. Please wait.
Restoring original required dependency versions: absl-py==1.4.0, fonttools==4.66.1, markdown==3.5.2, markupsafe==3.0.3, tzdata==2026.4, wrapt==2.5.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.1/130.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.5/126.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.9/103.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.5/347.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.3/248.3 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.5.1
    Uninstalling wrapt-2.5.1:
      Successfully uninstalled wrapt-2.5.1
  Attempting uninstall: tzdata
    Found existing

## 3. Run, resume or reconnect
Progress identifies the station, model, seed and epoch/round checkpoint. RUNNING confirms the worker is present, not that a specific fit has completed. Stopping this display leaves the worker running. A FAILED message retains the log; inspect its cause before retrying.

In [4]:
from airsense_r.notebook_job import run_and_watch
command = [sys.executable, str(RECOVERY_SCRIPT),
           "--root", str(ROOT), "--model-directory", str(MODEL_DIRECTORY),
           "--remote", REMOTE]
if TEST_MODE:
    import json
    command = json.loads(os.environ["AIRSENSE_P7_TEST_COMMAND"])
run_and_watch(command, JOB_DIRECTORY, ROOT)


RUNNING: attached to the job. Rerunning this cell reconnects to the same job.
INFO:root:Recovery: checking original source and persistent checkpoints
INFO:root:Recovery: required dependency versions match; checking full scientific identity
2026-10-21 06:28:43.092681: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
INFO:airsense_r.checkpoints:Checkpoint remotely confirmed: 63b4c4887f66b2b72f671694711dca1ca202f7d5
RUNNING: worker still active; waiting for the next progress checkpoint.
INFO:airsense_r.checkpoints:Checkpoint remotely confirmed: 63b4c4887f66b2b72f671694711dca1ca202f7d5
INFO:airsense_r.checkpoints:Checkpoint remotely confirmed: 63b4c4887f66b2b72f671694711dca1ca202f7d5
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress checkpoint.
RUNNING: worker still active; waiting for the next progress c

{'finished': 1792574421.9572122,
 'identity': '341dce051f1b60a9595bd1648730444c91812691ff0a2e13e19b884d1320246f',
 'pid': 5304,
 'return_code': 0,
 'started': 1792564106.2742786,
 'state': 'complete'}

## 4. Inspect completion
This runs only after the worker exits successfully. Cross-site generalization is not a causal site-shift penalty or inter-city transfer.

In [5]:
import json
from airsense_r.artifacts import RunStore, provenance
from airsense_r.notebook_job import status
if status(JOB_DIRECTORY)["state"] != "complete":
    raise RuntimeError("P7 is not finished. Rerun the preceding cell to reconnect or resume.")
output = ROOT / ".tmp/checkpoint-checkouts" / RUN_ID / "results" / RUN_ID
manifest = json.loads((output / "manifest.json").read_bytes())
store = RunStore(output, manifest["provenance"])
expected = provenance(ROOT, {})
for key in ("source_hash", "code_commit", "config_hash", "protocol_hash", "dataset_sha256"):
    if manifest["provenance"].get(key) != expected[key]:
        raise RuntimeError(f"Saved P7 results have incompatible {key}; preserve them for review")
parameters = manifest["provenance"].get("parameters", {})
storage_id = json.loads((MODEL_DIRECTORY / "storage.json").read_bytes())["storage_id"]
if (parameters.get("phase") != "P7" or parameters.get("run_id") != RUN_ID
        or parameters.get("model_storage_id") != storage_id):
    raise RuntimeError("Saved P7 results belong to a different run or persistent storage")
completion = store.load_json("complete.json")
if completion.get("execution_complete") is not True:
    raise RuntimeError("P7 execution completion record is missing")
display(completion)
display(store.load_json("site-shift-report.json"))
print("P7 EXECUTION COMPLETE. Save this executed notebook separately for independent review.")


{'execution_complete': True,
 'held_out_stations': ['Aotizhongxin',
  'Changping',
  'Dingling',
  'Dongsi',
  'Guanyuan',
  'Gucheng',
  'Huairou',
  'Nongzhanguan',
  'Shunyi',
  'Tiantan',
  'Wanliu',
  'Wanshouxigong'],
 'phase': 'P7',
 'report_sha256': 'e9a7d7287b362deb447d002034f56bdd67fe8f66c597314ac00a30c7880e926a',
 'scientifically_accepted': False}

{'estimand': 'cross_site_generalization_not_causal_penalty',
 'fixed_family_metrics': [{'mae': 10.301924390547663,
   'model': 'gru',
   'r2': 0.9407795788065593,
   'rmse': 19.410567296770306},
  {'mae': 10.946975636492253,
   'model': 'mlp',
   'r2': 0.945126747041126,
   'rmse': 19.12888274284781},
  {'mae': 10.279711903888566,
   'model': 'persistence',
   'r2': 0.9422164516034349,
   'rmse': 19.65486295419703},
  {'mae': 9.749186180871254,
   'model': 'xgboost',
   'r2': 0.9456180469392997,
   'rmse': 19.091455091937018}],
 'mean_station_selection_regret': {'ci_high': 0.7387928892039342,
  'ci_low': 0.40041729870083614,
  'estimate': 0.5503810810785558},
 'phase': 'P7',
 'policy_minus_fixed_family': {'gru': {'ci_high': 0.19190697846868524,
   'ci_low': -0.24441948732009833,
   'estimate': -0.022212486659096992},
  'mlp': {'ci_high': -0.46481470917560497,
   'ci_low': -0.8772843236909178,
   'estimate': -0.6672637326036853},
  'persistence': {'ci_high': 0.0, 'ci_low': 0.0, 'estimat

P7 EXECUTION COMPLETE. Save this executed notebook separately for independent review.
